In [57]:
import pandas as pd
import numpy as np
import streamlit as st
import plotly
import plotly.express as px


In [41]:
gapminder = pd.read_csv('C:/Users/jmcca/OneDrive/Documents/CUNY/Data Visualization/Homework/assignment_3/gapminder_data.tsv', 
                        sep='\t')

In [42]:
gapminder

,country,continent,year,lifeExp,pop,gdpPercap
0,Afghanistan,Asia,1952,28.801,8425333,779.445314
1,Afghanistan,Asia,1957,30.332,9240934,820.853030
2,Afghanistan,Asia,1962,31.997,10267083,853.100710
3,Afghanistan,Asia,1967,34.020,11537966,836.197138
4,Afghanistan,Asia,1972,36.088,13079460,739.981106
...,...,...,...,...,...,...
1699,Zimbabwe,Africa,1987,62.351,9216418,706.157306
1700,Zimbabwe,Africa,1992,60.377,10704340,693.420786
1701,Zimbabwe,Africa,1997,46.809,11404948,792.449960
1702,Zimbabwe,Africa,2002,39.989,11926563,672.038623


In [43]:
# check unique years
gapminder['year'].unique()

array([1952, 1957, 1962, 1967, 1972, 1977, 1982, 1987, 1992, 1997, 2002,
       2007], dtype=int64)

In [44]:
gapminder['country'].unique()

array(['Afghanistan', 'Albania', 'Algeria', 'Angola', 'Argentina',
       'Australia', 'Austria', 'Bahrain', 'Bangladesh', 'Belgium',
       'Benin', 'Bolivia', 'Bosnia and Herzegovina', 'Botswana', 'Brazil',
       'Bulgaria', 'Burkina Faso', 'Burundi', 'Cambodia', 'Cameroon',
       'Canada', 'Central African Republic', 'Chad', 'Chile', 'China',
       'Colombia', 'Comoros', 'Congo, Dem. Rep.', 'Congo, Rep.',
       'Costa Rica', "Cote d'Ivoire", 'Croatia', 'Cuba', 'Czech Republic',
       'Denmark', 'Djibouti', 'Dominican Republic', 'Ecuador', 'Egypt',
       'El Salvador', 'Equatorial Guinea', 'Eritrea', 'Ethiopia',
       'Finland', 'France', 'Gabon', 'Gambia', 'Germany', 'Ghana',
       'Greece', 'Guatemala', 'Guinea', 'Guinea-Bissau', 'Haiti',
       'Honduras', 'Hong Kong, China', 'Hungary', 'Iceland', 'India',
       'Indonesia', 'Iran', 'Iraq', 'Ireland', 'Israel', 'Italy',
       'Jamaica', 'Japan', 'Jordan', 'Kenya', 'Korea, Dem. Rep.',
       'Korea, Rep.', 'Kuwait', 'Leba

In [60]:
gapminder['country'][gapminder['continent'] == 'Oceania'].unique()

array(['Australia', 'New Zealand'], dtype=object)

# Dashboard Element Testing

In [45]:
continents = st.sidebar.multiselect(
    "Continent", gapminder["continent"].unique(), default=list(gapminder["continent"].unique())
)

min_year, max_year = int(gapminder["year"].min()), int(gapminder["year"].max())
if min_year < max_year:
    year_range = st.sidebar.slider("Year", min_year, max_year, (min_year, max_year))
else:
    year_range = (min_year, max_year)   # slider errors if min == max

min_val, max_val = float(gapminder["lifeExp"].min()), float(gapminder["lifeExp"].max())

value_range = st.sidebar.slider("lifeExp", min_val, max_val, (min_val, max_val))

# Apply filters
mask = (
    gapminder["continent"].isin(continents)
    & gapminder["year"].between(*year_range)
    & gapminder["lifeExp"].between(*value_range)
)
df = gapminder[mask]

In [46]:
# mean life expectancy Peru, 2007
df['lifeExp'][(df['year'] == 2007) & (df['country'] == 'Peru')].mean()

71.421

In [32]:
# mean life expectancy across all countries, 2007
df['lifeExp'][(df['year'] == 2007)].mean()

67.00742253521126

In [30]:
df['pop'][(df['year'] == 2007) & (df['country'] == 'Peru')]

1211    28674757
Name: pop, dtype: int64

In [31]:
df['gdpPercap'][(df['year'] == 2007) & (df['country'] == 'Peru')]

1211    7408.905561
Name: gdpPercap, dtype: float64

In [23]:
df[df['year']==2007].groupby('continent').agg(avg_life_expectancy=('lifeExp', 'mean'))

,avg_life_expectancy
continent,
Africa,54.806038
Americas,73.608120
Asia,70.728485
Europe,77.648600
Oceania,80.719500


In [58]:

import plotly.graph_objects as go

# Filter data
data_2007 = df[df["year"] == 2007]
peru = df[df["country"] == "Peru"].sort_values("year")

# Continent colors
continent_colors = {
    "Asia": "#4C78A8",
    "Europe": "#F2CF5B",
    "Africa": "#E45756",
    "Americas": "#72B7B2",
    "Oceania": "#B279A2"
}

fig = go.Figure()

# Scale bubble sizes by population
sizeref = 2.0 * data_2007["pop"].max() / (100**2)

# Countries in 2007, colored by continent
for continent, color in continent_colors.items():
    continent_df = data_2007[
        data_2007["continent"] == continent
    ]

    fig.add_trace(go.Scatter(
        x=continent_df["gdpPercap"],
        y=continent_df["lifeExp"],
        mode="markers",
        name=continent,
        marker=dict(
            size=continent_df["pop"],
            sizemode="area",
            sizeref=sizeref,
            sizemin=4,
            color=color,
            opacity=0.30,
            line=dict(width=0.5, color="white")
        ),
        customdata=continent_df[["country", "pop"]],
        hovertemplate=(
            "<b>%{customdata[0]}</b><br>"
            "GDP per capita: $%{x:,.0f}<br>"
            "Life expectancy: %{y:.1f}<br>"
            "Population: %{customdata[1]:,.0f}"
            "<extra></extra>"
        )
    ))

# Peru's historical path
fig.add_trace(go.Scatter(
    x=peru["gdpPercap"],
    y=peru["lifeExp"],
    mode="lines+markers",
    name="Peru",
    line=dict(color="black", width=1.5),
    marker =dict(
                size=peru["pop"],
                sizemode="area",
                sizeref=sizeref,
                sizemin=4,
                color="#FF00BF",
                opacity=1,
                line=dict(width=0.5, color="white")
            ),
    customdata=peru[["year", "pop"]],
    hovertemplate=(
        "<b>Peru</b><br>"
        "Year: %{customdata[0]}<br>"
        "GDP per capita: $%{x:,.0f}<br>"
        "Life expectancy: %{y:.1f}<br>"
        "Population: %{customdata[1]:,.0f}"
        "<extra></extra>"
    )
))

fig.update_layout(
    title="Peru's Development Compared to the World (2007)",
    xaxis_title="GDP per capita ($)",
    yaxis_title="Life Expectancy (years)",
    template="plotly_white",
    legend=dict(x=0.02, y=0.98),
    height=650
)

tick_vals = np.logspace(2, 5, 8)

fig.update_xaxes(
    type="log",
    tickmode="array",
    tickvals=tick_vals,
    ticktext=[f"${x:,.0f}" for x in tick_vals]
)

#fig.update_xaxes(type="log")

fig.show()


In [25]:
yearly = df.groupby(["year", "continent"], as_index=False)["lifeExp"].mean()
line_fig = px.line(yearly, 
                   x="year", 
                   y="lifeExp", 
                   color="continent", 
                   markers=True)
#line_fig.update_xaxes(dtick=5)   # whole-number year ticks, no 2019.5
line_fig.update_yaxes(
    dtick=None,
    nticks=8
)